# 📊 Notebook 01: Análisis Exploratorio (EDA)

**Proyecto:** Predicción de duración de casos judiciales cerrados en Bolivia  
**Autores:** Vergara & Patiño  
**Materia:** Tecnologías Emergentes (TI26)  

Este notebook realiza el análisis exploratorio de datos (EDA) sobre el dataset de casos cerrados del Ministerio Público de Bolivia.

In [ ]:
# Instalar dependencias
!pip install -q pyspark ipywidgets seaborn

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, datediff, when, lit, avg, count
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Iniciar Spark
spark = SparkSession.builder.master("local[*]").appName("EDA").getOrCreate()
print("Spark iniciado correctamente ✅")

## 1. Carga de Datos

In [ ]:
# Ruta del archivo (ruta local)
file_path = "../data/raw/CASOS_CERRADOS _publico__1.csv"
df = spark.read.csv(file_path, header=True, inferSchema=False)df = spark.read.csv(file_path, header=True, inferSchema=False)

print(f"Total de registros cargados: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")
print(f"\nColumnas: {df.columns}")

In [ ]:
# Vista previa de los datos
df.show(5, truncate=False)

## 2. Limpieza de Fechas y Cálculo de Duración

In [ ]:
# Limpiar fechas: solo conservar registros con formato válido
date_pattern = r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}$'
df_fechas = df.filter(
    col("denuncia_fecha_hora").rlike(date_pattern) &
    col("fecha_hora_cierre").rlike(date_pattern)
)

df_clean = df_fechas.withColumn(
    "denuncia_ts", to_timestamp(col("denuncia_fecha_hora"), "dd-MM-yyyy HH:mm:ss")
).withColumn(
    "cierre_ts", to_timestamp(col("fecha_hora_cierre"), "dd-MM-yyyy HH:mm:ss")
).withColumn(
    "duracion_dias", datediff("cierre_ts", "denuncia_ts")
).filter(col("duracion_dias") >= 0)

print(f"Registros válidos después de limpieza: {df_clean.count()}")

In [ ]:
# Cast de columnas numéricas
for col_name in ["victima_edad", "denunciado_edad", "hecho_gestion", "hecho_mes", "hecho_dia", "hecho_dia_semana"]:
    df_clean = df_clean.withColumn(col_name + "_num", col(col_name).cast("double"))

print("Columnas numéricas creadas ✅")

## 3. Estadísticas Descriptivas

In [ ]:
# Estadísticas de la variable objetivo (duración en días)
df_clean.select("duracion_dias").describe().show()

# Estadísticas de edades
df_clean.select("victima_edad_num", "denunciado_edad_num").describe().show()

## 4. Gráficos de Exploración

### 4.1 Distribución de Edades (Víctimas y Denunciados)

In [ ]:
# 1. Distribución de edades
df_filtered_age = df_clean.filter(
    ((col('victima_edad_num') > 0) & (col('victima_edad_num') <= 100)) |
    ((col('denunciado_edad_num') > 0) & (col('denunciado_edad_num') <= 100))
)
pandas_sample = df_filtered_age.select('victima_edad_num', 'denunciado_edad_num').sample(0.1).toPandas()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(pandas_sample[pandas_sample['victima_edad_num'] > 0]['victima_edad_num'].dropna(), bins=20, kde=True, color='steelblue')
plt.title('Distribución de Edad - Víctimas', fontsize=13)
plt.xlabel('Edad')
plt.ylabel('Frecuencia')

plt.subplot(1, 2, 2)
sns.histplot(pandas_sample[pandas_sample['denunciado_edad_num'] > 0]['denunciado_edad_num'].dropna(), bins=20, kde=True, color='coral')
plt.title('Distribución de Edad - Denunciados', fontsize=13)
plt.xlabel('Edad')
plt.ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('edades_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado: edades_distribucion.png ✅")

### 4.2 Duración Promedio por Mes del Hecho

In [ ]:
# 2. Duración promedio por mes
meses_duracion = df_clean.groupBy("hecho_mes_num").agg(
    avg("duracion_dias").alias("promedio_dias")
).orderBy("hecho_mes_num").toPandas()

meses_duracion['Mes'] = meses_duracion['hecho_mes_num'].map({
    1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'
})

plt.figure(figsize=(10, 6))
sns.barplot(data=meses_duracion.dropna(), x='Mes', y='promedio_dias', palette='coolwarm')
plt.title('Duración Promedio de Casos por Mes del Hecho', fontsize=14)
plt.xlabel('Mes')
plt.ylabel('Días promedio')
plt.tight_layout()
plt.savefig('duracion_meses.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado: duracion_meses.png ✅")

### 4.3 Duración Promedio por Departamento

In [ ]:
# 3. Duración por departamento
geo_stats = df_clean.groupBy("hecho_departamento").agg(
    avg("duracion_dias").alias("promedio_dias"),
    count("*").alias("num_casos")
).orderBy("promedio_dias", ascending=False).toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=geo_stats.dropna(), x='promedio_dias', y='hecho_departamento', palette='viridis')
plt.title('Duración Promedio de Casos por Departamento', fontsize=14)
plt.xlabel('Días promedio')
plt.ylabel('Departamento')
plt.tight_layout()
plt.savefig('duracion_departamento.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado: duracion_departamento.png ✅")

### 4.4 Correlaciones

In [ ]:
# 4. Correlaciones entre edades y duración
corr_v = df_clean.stat.corr("victima_edad_num", "duracion_dias")
corr_d = df_clean.stat.corr("denunciado_edad_num", "duracion_dias")

print(f"📈 Correlación edad víctima vs duración:    {corr_v:.4f}")
print(f"📈 Correlación edad denunciado vs duración:  {corr_d:.4f}")

if abs(corr_v) < 0.1 and abs(corr_d) < 0.1:
    print("\n⚠️ Las correlaciones son débiles, lo cual indica que la edad por sí sola")
    print("   no es un predictor fuerte de la duración del caso.")
else:
    print("\n✅ Se detecta cierta relación entre las edades y la duración.")

In [ ]:
## 5. Resumen del EDA

| Aspecto | Hallazgo |
|---|---|
| Registros totales | 161,642 |
| Variable objetivo | `duracion_dias` (días entre denuncia y cierre) |
| Edades | La mayoría de denunciados están entre 20-50 años |
| Mes | Hay variación estacional en la duración |
| Departamento | Diferencias significativas entre departamentos |
| Correlaciones | Débiles entre edad y duración || Correlaciones | Débiles entre edad y duración |

In [ ]:
# Cerrar sesión Spark
spark.stop()
print("\n🏁 Spark detenido. EDA completado exitosamente.")